# Automated Sales Email Report

## Overview

This project automates a simple sales-reporting workflow with Python. It reads sales transactions from a CSV file, calculates summary metrics, exports the summary to Excel, and emails the Excel report as an attachment using Gmail SMTP.

## Workflow
1. Load `sales_data.csv` with Pandas.
2. Calculate total sales, transaction count, average sale, and best-selling product.
3. Save the summary as `sales_report.xlsx`.
4. Build an email with the report attached.
5. Send the email through Gmail SMTP and close the connection.

## Requirements
- Python
- pandas
- openpyxl (for Excel export)
- A Gmail account configured to use an App Password

Install dependencies with `pip install pandas openpyxl`.

## Before running
Place `sales_data.csv` in the same folder as this notebook. Set the `EMAIL_ADDRESS`, `EMAIL_APP_PASSWORD`, and `EMAIL_RECIPIENT` environment variables, or enter them when prompted. Use a Gmail App Password rather than your normal account password. Never commit credentials to GitHub.

Running the final cell sends a real email to the configured recipient.

## 1. Import libraries and define file paths

In [ ]:
import os
import smtplib
from email.message import EmailMessage
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "sales_data.csv"
REPORT_FILE = BASE_DIR / "sales_report.xlsx"

## 2. Load and validate the sales data

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_FILE.name}. Place it in the notebook folder."
    )

sales = pd.read_csv(DATA_FILE)
required_columns = {"Product", "Sales"}
missing_columns = required_columns.difference(sales.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

sales["Sales"] = pd.to_numeric(sales["Sales"], errors="raise")
if sales.empty:
    raise ValueError("The sales data contains no transactions.")

sales.head()

## 3. Calculate report metrics

In [ ]:
total_sales = sales["Sales"].sum()
total_transactions = sales["Sales"].count()
average_sale = sales["Sales"].mean()
product_sales = sales.groupby("Product")["Sales"].sum()
best_product = product_sales.idxmax()

report = pd.DataFrame({
    "Metric": [
        "Total Sales",
        "Total Transactions",
        "Average Sale",
        "Best-Selling Product",
    ],
    "Value": [
        total_sales,
        total_transactions,
        average_sale,
        best_product,
    ],
})

report

## 4. Export the report to Excel

In [ ]:
report.to_excel(REPORT_FILE, index=False)
print(f"Report saved to: {REPORT_FILE}")

## 5. Prepare the email

In [ ]:
email_address = os.getenv("EMAIL_ADDRESS") or input("Your Gmail address: ").strip()
recipient = os.getenv("EMAIL_RECIPIENT") or input("Recipient email address: ").strip()
email_password = os.getenv("EMAIL_APP_PASSWORD")
if not email_password:
    from getpass import getpass
    email_password = getpass("Your Gmail App Password: ")

if not email_address or not recipient or not email_password:
    raise ValueError("Email address, recipient, and App Password are required.")

message = EmailMessage()
message["From"] = email_address
message["To"] = recipient
message["Subject"] = "Automated Sales Report"
message.set_content(
    "Hello,\n\n"
    "Here is your automated sales summary:\n\n"
    f"Total Sales: {total_sales:,.2f}\n"
    f"Total Transactions: {total_transactions}\n"
    f"Average Sale: {average_sale:,.2f}\n"
    f"Best-Selling Product: {best_product}\n\n"
    "Please find the full Excel report attached.\n\n"
    "Regards"
)

message.add_attachment(
    REPORT_FILE.read_bytes(),
    maintype="application",
    subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    filename=REPORT_FILE.name,
)
print(f"Email prepared for: {recipient}")

## 6. Send the email

In [ ]:
# Running this cell sends a real email to the configured recipient.
with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as server:
    server.starttls()
    server.login(email_address, email_password)
    server.send_message(message)

print(f"Email sent successfully to {recipient}.")